In [1]:
!wget https://github.com/kairess/toy-datasets/raw/master/dog-inflammation.zip
!unzip -q dog-eye-inflammation.zip

--2023-10-03 16:13:38--  https://github.com/kairess/toy-datasets/raw/master/dog-inflammation.zip
github.com (github.com) 해석 중... 20.200.245.247
다음으로 연결 중: github.com (github.com)|20.200.245.247|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 302 Found
위치: https://raw.githubusercontent.com/kairess/toy-datasets/master/dog-inflammation.zip [따라감]
--2023-10-03 16:13:39--  https://raw.githubusercontent.com/kairess/toy-datasets/master/dog-inflammation.zip
raw.githubusercontent.com (raw.githubusercontent.com) 해석 중... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
다음으로 연결 중: raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 4592449 (4.4M) [application/zip]
저장 위치: `dog-inflammation.zip.3'

dog-inflammation.zi 100%[===================>]   4.38M  3.57MB/s    /  1.2s    

2023-10-03 16:13:41 (3.57 MB/s) - `dog-inflammation.zip.3' 저장함 [4592449/4592449]

unzip:  cannot find or open dog-eye-inflammation.zip, dog-eye-infla

In [2]:
!unzip -q Project/Python/Dog AI/dog-inflammation.zip

unzip:  cannot find or open Project/Python/Dog, Project/Python/Dog.zip or Project/Python/Dog.ZIP.


In [ ]:
import os

In [ ]:
!ls

In [ ]:
ls

In [ ]:
!unzip -q dog-inflammation.zip

In [ ]:
from fastai.vision.all import *
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

In [ ]:
#from fastai.vision.all import *

path = './dog-inflammation/'

block = DataBlock(
    blocks = (ImageBlock, CategoryBlock),
    get_items = get_image_files,
    get_y = parent_label,
)

loader = block.dataloaders('./dog-inflammation/')

loader.show_batch()

In [ ]:
learn = vision_learner(loader, resnet18, metrics=accuracy)

learn.fine_tune(epochs = 3)

In [ ]:
learn.show_results()

In [ ]:
neg_path = './dog-inflammation/Negative/D0_03f7e7c0-60a5-11ec-8402-0a7404972c70.jpg'
pos_path = './dog-inflammation/Positive/D0_02fa7d26-60a5-11ec-8402-0a7404972c70.jpg'

test_loader = loader.test_dl([neg_path, pos_path])

neg_x, pos_x = next(iter(test_loader))[0]

neg_x = neg_x.unsqueeze(0)
pos_x = pos_x.unsqueeze(0)

print(neg_x.shape, pos_x.shape)

neg_img = Image.open(neg_path)
pos_img = Image.open(pos_path)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(neg_img)
ax[0].axis('off')
ax[1].imshow(pos_img)
ax[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#import torch
#import torch.nn as nn
#import torchvision.models as models
#import matplotlib.pyplot as plt
#from torchvision import transforms
#from PIL import Image

# 이미지를 모델의 입력 형식에 맞게 전처리하는 변환 함수 정의
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 모델 정의 (예: ResNet-50)
model = models.resnet50(pretrained=True)
model.eval()

# 이미지 열기
image_path = './dog-inflammation/Positive/D0_02fa7d26-60a5-11ec-8402-0a7404972c70.jpg'
img = Image.open(image_path)

# 이미지 전처리
input_tensor = preprocess(img)
input_batch = input_tensor.unsqueeze(0)  # 배치 차원 추가

# 모델 예측 수행
with torch.no_grad():
    output = model(input_batch)

# 가장 높은 확률을 갖는 클래스의 인덱스 가져오기
predicted_class = torch.argmax(output).item()

# 모델의 마지막 컨볼루션 레이어 가져오기
final_conv_layer = model.layer4[-1]

# 모델의 그래디언트 설정
model.zero_grad()

# 예측된 클래스에 대한 그래디언트 계산
output[0, predicted_class].backward()

# 클래스별 중요도를 가중치로 사용하여 활성화 맵 생성
grads = final_conv_layer.weight.grad
weights = torch.mean(grads, dim=(2, 3))[0]

# 활성화 맵을 입력 이미지 크기로 업샘플링
activation_map = torch.zeros(weights.shape[1:], dtype=torch.float32)
for i, w in enumerate(weights):
    activation_map += w * final_conv_layer.weight[i]

# 활성화 맵을 시각화
activation_map = activation_map.cpu().detach().numpy()
activation_map = (activation_map - activation_map.min()) / (activation_map.max() - activation_map.min())
plt.imshow(activation_map, cmap='jet', alpha=0.7)
plt.axis('off')
plt.show()